In [1]:
import os
import time
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI


In [2]:
# Load environment variables from current directory .env and override existing ones
load_dotenv(".env", override=True)

# load api key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI(api_key=api_key)
MAX_RETRIES = 6
MODEL_ID = "gpt-4.1-2025-04-14"



In [3]:
# Path to the summarized principles JSON

TRESHOLD = .7

principles_path = Path(r"C:\Users\laras\OneDrive\Dokumente\duke_classes\icai_project\Principle-Elicitation\DecontextualPrinciples\self_written_ai_principles_th=0.7_summarized.json")

evaluator_prompt_path = Path("judge_few_shot_examples_dimensions.txt")

with principles_path.open("r", encoding="utf-8") as f:
    principles_raw = json.load(f)

if isinstance(principles_raw, list):
    principles_data = {str(i): item for i, item in enumerate(principles_raw)}
else:
    principles_data = principles_raw

with evaluator_prompt_path.open("r", encoding="utf-8") as f:
    evaluator_prompt = f.read()

print("Loaded principles from:", principles_path)
print("Loaded evaluator prompt from:", evaluator_prompt_path)



Loaded principles from: C:\Users\laras\OneDrive\Dokumente\duke_classes\icai_project\Principle-Elicitation\DecontextualPrinciples\self_written_ai_principles_th=0.7_summarized.json
Loaded evaluator prompt from: judge_few_shot_examples_dimensions.txt


In [4]:
evaluator_prompt

'# Identity\n\nYou are an impartial evaluator that scores how well principles conform to seven evaluative dimensions.\n\nInstructions\nRate each principle based on the dimensions listed below.\nProvide a separate Likert scale rating that indicates how well the principle agrees with each dimension, based on the following scale:\n\n1 (Strongly disagree)\n2 (Disagree)\n3 (Neither agree nor disagree)\n4 (Agree)\n5 (Strongly agree)\n\nIn your response, only specify the dimension name and numerical rating. Do not elaborate or justify the ratings.\n\n# Dimensions\n\n[Moral Grounding] The principle is informed by Moral Grounding.\n\n[Fairness] The principle treats all affected parties equally and without bias.\n\n[Generality] The principle is broadly applicable across different contexts.\n\n[Clarity] The wording and intent of the principle are interpretable and unambiguous.\n\n[Applicability] The principle can feasibly be actioned upon and be achieved by the AI in a real-world context.\n\n[Con

In [5]:
def parse_dimension_scores(raw_text: str):
    scores = {}
    for line in raw_text.strip().splitlines():
        m = re.match(r"\s*\[(.*?)\]\s*(?:--|:)\s*(\d+)", line)
        if m:
            dim_name = m.group(1).strip()
            score = int(m.group(2))
            scores[dim_name] = score
    return scores



In [6]:
results = []
total = len(principles_data)
processed = 0

for key, item in principles_data.items():
    cluster_id = item.get("cluster_id", key)
    summarized_principle = item.get("summarized_principle", "").strip()

    if not summarized_principle:
        continue

    user_content = (
        "Principle evaluated:\n"
        f"{summarized_principle}\n\n"
        "Evaluator output:"
    )

    retries = 0
    while True:
        try:
            resp = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": evaluator_prompt},
                    {"role": "user", "content": user_content},
                ],
                max_tokens=256,
                temperature=0,
            )
            judge_text = resp.choices[0].message.content
            break
        except Exception as e:
            retries += 1
            if retries > MAX_RETRIES:
                raise
            print(f"Error ({e}), retry {retries}...")
            time.sleep(min(30, 2 ** retries))

    scores = parse_dimension_scores(judge_text)
    results.append(
        {
            "cluster_id": cluster_id,
            "summarized_principle": summarized_principle,
            "raw_judge_output": judge_text,
            "scores": scores,
        }
    )

    processed += 1

    if processed % 50 == 0:
        print(f"Scored {processed} / {total} principles so far...")

print(f"Evaluated {len(results)} principles (out of {total}).")



Evaluated 25 principles (out of 25).


In [7]:
rows = []
for r in results:
    base = {
        "cluster_id": r["cluster_id"],
        "summarized_principle": r["summarized_principle"],
    }
    for dim, score in r["scores"].items():
        base[dim] = score
    rows.append(base)

df = pd.DataFrame(rows)

non_dim_cols = {"cluster_id", "summarized_principle"}
score_cols = [c for c in df.columns if c not in non_dim_cols]

df["average_score"] = df[score_cols].mean(axis=1)

for r, avg in zip(results, df["average_score"]):
    r["average_score"] = float(avg)

output_json_path = Path(f"principle_dimension_scores_self_written_ai_principles_tH={TRESHOLD}.json")
with output_json_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved results (including average_score) to:", output_json_path)


Saved results (including average_score) to: principle_dimension_scores_self_written_ai_principles_tH=0.7.json


In [48]:
path = "principle_dimension_scores_self_written_ai_principles_tH=0.7.json"
with open(path, "r", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))
df.sort_values(by='average_score',ascending=False)

,cluster_id,summarized_principle,raw_judge_output,scores,average_score
0,0,**Summarized Principle:**\n\nAlways provide ac...,[Moral Grounding] -- 5 (Strongly agree) \n[Fa...,"{'Moral Grounding': 5, 'Fairness': 5, 'General...",4.714286
1,1,**Summarized Principle:**\n\nThe AI language m...,[Moral Grounding] -- 5 (Strongly agree) \n[Fa...,"{'Moral Grounding': 5, 'Fairness': 5, 'General...",4.428571
10,10,"The AI model should prioritize precision, reli...",[Moral Grounding] -- 4 (Agree) \n[Fairness] -...,"{'Moral Grounding': 4, 'Fairness': 4, 'General...",4.428571
18,18,**Summarized Principle:** \nThe AI should be ...,[Moral Grounding] -- 4 (Agree) \n[Fairness] -...,"{'Moral Grounding': 4, 'Fairness': 5, 'General...",4.285714
2,2,**Summarized Principle:**\n\nAlways be helpful...,[Moral Grounding] -- 4 (Agree) \n[Fairness] -...,"{'Moral Grounding': 4, 'Fairness': 4, 'General...",4.142857
9,9,**Summarized Principle:** \nAlways provide so...,[Moral Grounding] -- 5 (Strongly agree) \n[Fa...,"{'Moral Grounding': 5, 'Fairness': 5, 'General...",4.142857
22,22,**Summarized Principle:** \nProvide age-appro...,[Moral Grounding] -- 4 (Agree) \n[Fairness] -...,"{'Moral Grounding': 4, 'Fairness': 4, 'General...",4.142857
4,4,**Summarized Principle:**\n\nFocus precisely o...,[Moral Grounding] -- 3 (Neither agree nor disa...,"{'Moral Grounding': 3, 'Fairness': 4, 'General...",4.142857
7,7,Summarized Principle:\nCommunicate information...,[Moral Grounding] -- 5 (Strongly agree) \n[Fa...,"{'Moral Grounding': 5, 'Fairness': 5, 'General...",4.142857
20,20,**Summarized Principle:** \nThe AI should ope...,[Moral Grounding] -- 3 (Neither agree nor disa...,"{'Moral Grounding': 3, 'Fairness': 4, 'General...",4.000000


In [49]:
print("\nTop 15 principles by average_score (AI principles):")
top15 = df.sort_values("average_score", ascending=False).head(15)
for _, row in top15.iterrows():
    print(f"[avg={row['average_score']:.2f}] (cluster_id={row['cluster_id']}) {row['summarized_principle']}")




Top 15 principles by average_score (AI principles):
[avg=4.71] (cluster_id=0) **Summarized Principle:**

Always provide accurate information by verifying facts with reliable, well-respected sources, and avoid spreading false, hateful, discriminatory, or plagiarized content. Ensure that responses do no harm and maintain credibility, especially in professional and academic contexts.
[avg=4.43] (cluster_id=1) **Summarized Principle:**

The AI language model should consistently provide honest, accurate, unbiased, and up-to-date information delivered in a clear, concise, and professional manner. It should be kind, respectful, and friendly in tone, while avoiding emotional or politically biased responses. The AI should communicate clearly, distinguish nuances in questions, and always present information from multiple viewpoints when relevant. Additionally, the model should excel in both informational and creative tasks, remaining helpful, compassionate, and as non-robotic as possible.
[avg=

In [50]:
print("\nBottom 15 principles by average_score (AI principles):")
bottom15 = df.sort_values("average_score", ascending=True).head(15)
for _, row in bottom15.iterrows():
    print(f"[avg={row['average_score']:.2f}] (cluster_id={row['cluster_id']}) {row['summarized_principle']}")


Bottom 15 principles by average_score (AI principles):
[avg=2.14] (cluster_id=23) **Summarized Principle:**  
Preference for both professional and storytelling modes, as talking to strangers is generally favored by most people.
[avg=2.14] (cluster_id=19) **Summarized Principle:**  
Always prioritize and fulfill user needs without limitation.
[avg=2.43] (cluster_id=13) Incorporate humor with emojis and use commas appropriately in long sentences.
[avg=2.43] (cluster_id=17) No particular preference or opinion.
[avg=3.00] (cluster_id=15) Summarized Principle:  
No opinion or input provided due to lack of experience.
[avg=3.00] (cluster_id=8) **Summarized Principle:**

The AI should be able to handle data analysis, problem-solving, and game playing (adapting to both professional and amateur skill levels), as well as provide real-time assistance during gameplay. It should function generally, focusing on personal interests like music and sports, without emphasizing work or educational suppor

In [55]:
path = "principle_dimension_scores_self_written_ai_principles_tH=0.42.json"

# Load JSON → DataFrame
with open(path, "r", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

# Sort by average_score descending and add rank (1 = highest score)
df_sorted = df.sort_values(by="average_score", ascending=False).reset_index(drop=True)
df_sorted["rank"] = df_sorted.index + 1

# Keep only desired columns
df_ranked = df_sorted[["cluster_id", "rank", "summarized_principle", "average_score"]]

# Save to JSON (list of dicts)
out_path = "ranked/principle_dimension_scores_self_written_ai_principles_th=0.5_ranked.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(df_ranked.to_dict(orient="records"), f, indent=2, ensure_ascii=False)

print("Saved ranked results to:", out_path)

Saved ranked results to: ranked/principle_dimension_scores_self_written_ai_principles_th=0.5_ranked.json
